In [19]:
import numpy as np
import onnx
import tvm
from tvm import relax
from tvm.relax.frontend.onnx import from_onnx
import time

In [43]:
onnx_model = onnx.load("../models/yolov8n.onnx")
mod = from_onnx(onnx_model, keep_params_in_input=False)
mod, params = relax.frontend.detach_params(mod)

In [44]:
mod

# from tvm.script import ir as I
# from tvm.script import tir as T
# from tvm.script import relax as R

@I.ir_module
class Module:
    @R.function
    def main(images: R.Tensor((1, 3, 640, 640), dtype="float32")) -> R.Tensor((1, 84, 8400), dtype="float32"):
        R.func_attr({"num_input": 1})
        with R.dataflow():
            lv: R.Tensor((1, 16, 320, 320), dtype="float32") = R.nn.conv2d(images, metadata["relax.expr.Constant"][0], strides=[2, 2], padding=[1, 1, 1, 1], dilation=[1, 1], groups=1, data_layout="NCHW", kernel_layout="OIHW", out_layout="NCHW", out_dtype="void")
            lv1: R.Tensor((1, 16, 1, 1), dtype="float32") = R.reshape(metadata["relax.expr.Constant"][1], R.shape([1, 16, 1, 1]))
            lv2: R.Tensor((1, 16, 320, 320), dtype="float32") = R.add(lv, lv1)
            lv3: R.Tensor((1, 16, 320, 320), dtype="float32") = R.sigmoid(lv2)
            lv4: R.Tensor((1, 16, 320, 320), dtype="float32") = R.multiply(lv2, lv3)
            lv5: R.Tensor((1, 32, 160, 16

In [45]:
params

{}

In [46]:
# target = tvm.target.Target("llvm")
target = target = tvm.target.Target({"kind": "llvm", "num-cores": 11})

with target:
    mod = relax.get_pipeline("default")(mod)

ex = tvm.compile(mod, target=target) # , relax_pipeline="zero"
dev = tvm.cpu(0)
vm = relax.VirtualMachine(ex, dev)

In [47]:
x = np.random.rand(1, 3, 640, 640).astype("float32")
x_tvm = tvm.runtime.tensor(x, dev)
# w_tvm = [tvm.runtime.tensor(p, dev) for p in params["main"]]

y = vm["main"](x_tvm) # *w_tvm
print(type(y))
if isinstance(y, tvm.ir.Array):
    print([o.shape for o in y])
else:
    print(y.shape)

<class 'tvm.runtime._tensor.Tensor'>
(1, 84, 8400)


In [52]:
def sync_tvm(dev):
    try:
        dev.sync()
    except Exception:
        pass

def bench_tvm(vm, dev, x_np, runs=10, warmup=20):
    x_tvm = tvm.runtime.tensor(x_np, dev)
    # w_tvm = [tvm.runtime.tensor(p, dev) for p in params["main"]]

    for _ in range(warmup):
        _ = vm["main"](x_tvm)
    sync_tvm(dev)

    t0 = time.perf_counter()
    for _ in range(runs):
        _ = vm["main"](x_tvm)
    sync_tvm(dev)
    t1 = time.perf_counter()

    ms = (t1 - t0) * 1000 / runs
    fps = 1000.0 / ms
    return ms, fps

In [53]:
import time
import torch
from ultralytics import YOLO

def sync_torch(device):
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()

@torch.inference_mode()
def bench_pt(pt_path="../models/yolov8.pt", imgsz=640, device="cpu", runs=10, warmup=20):
    y = YOLO(pt_path)
    model = y.model.eval().to(device)

    x = torch.randn(1, 3, imgsz, imgsz, device=device, dtype=torch.float32)

    for _ in range(warmup):
        _ = model(x)
    sync_torch(torch.device(device))

    t0 = time.perf_counter()
    for _ in range(runs):
        _ = model(x)
    sync_torch(torch.device(device))
    t1 = time.perf_counter()

    ms = (t1 - t0) * 1000 / runs
    fps = 1000.0 / ms
    return ms, fps


In [54]:
x_np = np.random.rand(1,3,640,640).astype("float32")
# TVM CPU
ms_tvm_cpu, fps_tvm_cpu = bench_tvm(relax.VirtualMachine(ex, dev), tvm.cpu(0), x_np)
# PyTorch CPU
ms_pt_cpu, fps_pt_cpu = bench_pt("yolov8n.pt", imgsz=640, device="cpu")

In [55]:
ms_pt_cpu, fps_pt_cpu

(66.75375829945551, 14.980429948438672)

In [56]:
ms_tvm_cpu, fps_tvm_cpu

(3208.750854199752, 0.3116477549795294)

In [57]:
import onnx, tvm
from tvm import relax
from tvm.relax.frontend.onnx import from_onnx

onnx_model = onnx.load("../models/yolov8n.onnx")
mod = from_onnx(onnx_model, keep_params_in_input=False)  # embedded weights

target = tvm.target.Target({"kind": "llvm", "num-cores": 11})

# Do NOT run relax.get_pipeline manually here
ex = tvm.compile(mod, target=target, relax_pipeline="default", tir_pipeline="default")

dev = tvm.cpu(0)
vm = relax.VirtualMachine(ex, dev)

import numpy as np
x_np = np.random.rand(1, 3, 640, 640).astype("float32")
x_tvm = tvm.runtime.tensor(x_np, dev)

ftimer = vm.time_evaluator("main", dev, number=5, repeat=10)
prof = ftimer(x_tvm)
print("TVM ms:", np.mean(prof.results) * 1000)


TVM ms: 3211.18006998


In [58]:
info = tvm.support.libinfo()
print("USE_LLVM:", info.get("USE_LLVM"))
print("CMAKE_BUILD_TYPE:", info.get("CMAKE_BUILD_TYPE"))


USE_LLVM: ON
CMAKE_BUILD_TYPE: None
